# Notebook 06 — Model Training & Evaluation
**Project:** Loan Default Risk Analysis and Prediction  
**Phase:** 6b of 8  
**Objective:** Train a Random Forest classifier and a Logistic Regression baseline, evaluate both models with multiple metrics, visualise ROC curves, Precision-Recall curves, confusion matrices, and feature importances. Save model artefacts.

---

## 0. Environment Setup

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split

from src.data_loader import load_data
from src.data_cleaner import clean_data
from src.feature_engineering import build_preprocessor, get_feature_names
from src.model import (
    train_model, train_baseline, evaluate_model, save_artefacts,
    plot_confusion_matrix, plot_roc_curve, plot_pr_curve,
    plot_feature_importance
)
from src.config import (
    RAW_DATA_PATH, TARGET_COLUMN, RANDOM_STATE, TEST_SIZE,
    DECISION_THRESHOLD, FIGURES_DIR, TABLES_DIR
)

sns.set_theme(style='whitegrid', font_scale=1.1)
plt.rcParams['figure.dpi'] = 110
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
TABLES_DIR.mkdir(parents=True, exist_ok=True)
print('Environment ready.')

---
## 1. Load Data & Prepare Train/Test Split

In [ ]:
df = clean_data(load_data(RAW_DATA_PATH))
X, y, preprocessor = build_preprocessor(df)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
)
print(f'Train: {X_train.shape[0]:,}   Test: {X_test.shape[0]:,}')
print(f'Default rate — Train: {y_train.mean()*100:.2f}%  Test: {y_test.mean()*100:.2f}%')

---
## 2. Train Logistic Regression Baseline

In [ ]:
from src.feature_engineering import build_preprocessor as _bp
_, _, prep_lr = _bp(df)   # fresh unfitted preprocessor for LR

print('Training Logistic Regression baseline...')
lr_pipeline = train_baseline(X_train, y_train, prep_lr)
print('Baseline training complete.')

In [ ]:
print('=== BASELINE: Logistic Regression ===')
lr_metrics = evaluate_model(lr_pipeline, X_test, y_test)

---
## 3. Train Random Forest Model

In [ ]:
print('Training Random Forest (200 trees, balanced weights)...')
print('This may take 1-3 minutes on the full dataset...')
rf_pipeline = train_model(X_train, y_train, preprocessor)
print('Random Forest training complete.')

---
## 4. Evaluate Random Forest

In [ ]:
print('=== MAIN MODEL: Random Forest ===')
rf_metrics = evaluate_model(rf_pipeline, X_test, y_test)

---
## 5. Model Comparison Table

In [ ]:
comparison = pd.DataFrame([
    {'Model': 'Logistic Regression (Baseline)',
     'ROC-AUC':        lr_metrics['roc_auc'],
     'Avg Precision':  lr_metrics['avg_precision'],
     'F1 (Default)':   lr_metrics['f1'],
     'Accuracy':       lr_metrics['accuracy']},
    {'Model': 'Random Forest',
     'ROC-AUC':        rf_metrics['roc_auc'],
     'Avg Precision':  rf_metrics['avg_precision'],
     'F1 (Default)':   rf_metrics['f1'],
     'Accuracy':       rf_metrics['accuracy']},
])
comparison.to_csv(TABLES_DIR / 'model_comparison.csv', index=False)
print('Saved -> outputs/tables/model_comparison.csv')
comparison

---
## 6. Confusion Matrix

In [ ]:
fig = plot_confusion_matrix(rf_pipeline, X_test, y_test)
fig.savefig(FIGURES_DIR / 'model_confusion_matrix.png', bbox_inches='tight')
plt.show()
print('Saved -> outputs/figures/model_confusion_matrix.png')

---
## 7. ROC Curve

In [ ]:
# Compare RF and LR on same ROC plot
from sklearn.metrics import roc_curve, roc_auc_score

fig, ax = plt.subplots(figsize=(7, 5))

for pipeline, label, color in [
    (rf_pipeline, 'Random Forest',              '#4a90d9'),
    (lr_pipeline, 'Logistic Regression',        '#e05c5c'),
]:
    y_prob = pipeline.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    auc = roc_auc_score(y_test, y_prob)
    ax.plot(fpr, tpr, color=color, linewidth=2, label=f'{label} (AUC={auc:.4f})')

ax.plot([0,1],[0,1],'k--',linewidth=1,alpha=0.5,label='Random Baseline')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curve Comparison', fontweight='bold')
ax.legend(loc='lower right')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'model_roc_curve.png', bbox_inches='tight')
plt.show()
print('Saved -> outputs/figures/model_roc_curve.png')

---
## 8. Precision-Recall Curve

In [ ]:
fig = plot_pr_curve(rf_pipeline, X_test, y_test)
fig.savefig(FIGURES_DIR / 'model_pr_curve.png', bbox_inches='tight')
plt.show()
print('Saved -> outputs/figures/model_pr_curve.png')

---
## 9. Feature Importances

In [ ]:
fig = plot_feature_importance(rf_pipeline)
fig.savefig(FIGURES_DIR / 'model_feature_importance.png', bbox_inches='tight')
plt.show()
print('Saved -> outputs/figures/model_feature_importance.png')

---
## 10. Threshold Analysis

In [ ]:
from sklearn.metrics import f1_score, precision_score, recall_score

y_prob = rf_pipeline.predict_proba(X_test)[:, 1]
thresholds = np.arange(0.1, 0.8, 0.05)

rows = []
for t in thresholds:
    y_pred = (y_prob >= t).astype(int)
    rows.append({
        'Threshold':  round(t, 2),
        'Precision':  round(precision_score(y_test, y_pred, zero_division=0), 4),
        'Recall':     round(recall_score(y_test, y_pred, zero_division=0), 4),
        'F1':         round(f1_score(y_test, y_pred, zero_division=0), 4),
    })

thresh_df = pd.DataFrame(rows)
thresh_df.to_csv(TABLES_DIR / 'threshold_analysis.csv', index=False)
print('Saved -> outputs/tables/threshold_analysis.csv')

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(thresh_df['Threshold'], thresh_df['Precision'], label='Precision', color='#4a90d9', marker='o', markersize=4)
ax.plot(thresh_df['Threshold'], thresh_df['Recall'],    label='Recall',    color='#e05c5c', marker='o', markersize=4)
ax.plot(thresh_df['Threshold'], thresh_df['F1'],        label='F1 Score',  color='#27ae60', marker='o', markersize=4)
ax.axvline(DECISION_THRESHOLD, color='k', linestyle='--', linewidth=1, label=f'Chosen threshold ({DECISION_THRESHOLD})')
ax.set_xlabel('Decision Threshold')
ax.set_ylabel('Score')
ax.set_title('Precision / Recall / F1 vs Decision Threshold', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'model_threshold_analysis.png', bbox_inches='tight')
plt.show()
print('Saved -> outputs/figures/model_threshold_analysis.png')

---
## 11. Probability Score Distribution

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))

ax.hist(y_prob[y_test == 0], bins=50, alpha=0.55, color='#4a90d9',
        label='No Default', density=True, edgecolor='none')
ax.hist(y_prob[y_test == 1], bins=50, alpha=0.55, color='#e05c5c',
        label='Default', density=True, edgecolor='none')
ax.axvline(DECISION_THRESHOLD, color='k', linestyle='--', linewidth=1.5,
           label=f'Threshold = {DECISION_THRESHOLD}')
ax.set_xlabel('Predicted Default Probability')
ax.set_ylabel('Density')
ax.set_title('Predicted Probability Distribution by Actual Label', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'model_prob_distribution.png', bbox_inches='tight')
plt.show()
print('Saved -> outputs/figures/model_prob_distribution.png')

---
## 12. Save Model Artefacts

In [ ]:
save_artefacts(rf_pipeline, preprocessor)
print('Model artefacts saved to models/')

---
## 13. Quick Inference Test

In [ ]:
from src.model import load_pipeline, predict_default

loaded = load_pipeline()
sample = X_test.iloc[:3].copy()
result = predict_default(loaded, sample)

sample_out = sample.copy()
sample_out['Actual']      = y_test.iloc[:3].values
sample_out['Probability'] = [round(p, 4) for p in result['probability']]
sample_out['Prediction']  = result['prediction']
sample_out['RiskLabel']   = result['risk_label']
sample_out[['Actual','Probability','Prediction','RiskLabel']]

---
## 14. Model Performance Summary

| Metric | Logistic Regression | Random Forest |
|---|---|---|
| ROC-AUC | *(fill after run)* | *(fill after run)* |
| Avg Precision | *(fill after run)* | *(fill after run)* |
| F1 (Default class) | *(fill after run)* | *(fill after run)* |
| Accuracy | *(fill after run)* | *(fill after run)* |
| Decision threshold | 0.40 | 0.40 |

---
**Next:** Phase 7 — Streamlit Dashboard